In [1]:
from pathlib import Path
import pandas as pd
import json

import sys
sys.path.append("../../../utils/")

from utils import *

In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[3]

NOMBRE_DATASET_ENTRADA = "CIC18__cleanning__v1"
NOMBRE_SPLIT = "CIC18__split__v1"

RUTA_DATASET_LIMPIO = PROJECT_ROOT / "02_datasets" / "processed_analisis_estadistico" / NOMBRE_DATASET_ENTRADA
RUTA_SALIDA = PROJECT_ROOT / "02_datasets" / "processed_analisis_estadistico" / NOMBRE_SPLIT

NOMBRE_DATASET_LIMPIO = f"{NOMBRE_DATASET_ENTRADA}.csv"
NOMBRE_TRAIN = f"{NOMBRE_SPLIT}__train.csv"
NOMBRE_TEST = f"{NOMBRE_SPLIT}__test.csv"
NOMBRE_REPORTE = f"{NOMBRE_SPLIT}_report.json"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"
TEST_SIZE = 0.20
RANDOM_STATE = 42

In [3]:
print("PROJECT_ROOT:")
print(PROJECT_ROOT)
print()

print("Dataset limpio de entrada:")
print(RUTA_DATASET_LIMPIO / NOMBRE_DATASET_LIMPIO)
print()

print("Ruta de salida:")
print(RUTA_SALIDA)

PROJECT_ROOT:
/LUSTRE/home/inginf/u32902122/TFG

Dataset limpio de entrada:
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/CIC18__cleanning__v1/CIC18__cleanning__v1.csv

Ruta de salida:
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/CIC18__split__v1


In [4]:
input_path = RUTA_DATASET_LIMPIO / NOMBRE_DATASET_LIMPIO

if not input_path.exists():
    raise FileNotFoundError(f"No existe el dataset limpio en: {input_path}")

df = cargar_dataset(nombre_dataset=NOMBRE_DATASET_LIMPIO, ruta_base=RUTA_DATASET_LIMPIO)

shape_original = df.shape

print("Forma del dataset limpio:")
print(shape_original)

Forma del dataset limpio:
(11982425, 51)


In [5]:
df.head()

,DST_PORT,PROTOCOL,FLOW_DURATION,TOT_FWD_PKTS,TOT_BWD_PKTS,TOTLEN_FWD_PKTS,FWD_PKT_LEN_MAX,FWD_PKT_LEN_MIN,FWD_PKT_LEN_MEAN,BWD_PKT_LEN_MAX,...,BWD_BLK_RATE_AVG,INIT_FWD_WIN_BYTS,INIT_BWD_WIN_BYTS,FWD_SEG_SIZE_MIN,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_MIN,LABEL
0,443,6,141385,9,7,553.0,202.0,0.0,61.444444,1460.0,...,0,8192,119,20,0.0,0.0,0.0,0.0,0.0,Benign
1,49684,6,281,2,1,38.0,38.0,0.0,19.000000,0.0,...,0,123,0,20,0.0,0.0,0.0,0.0,0.0,Benign
2,443,6,279824,11,15,1086.0,385.0,0.0,98.727273,1460.0,...,0,8192,1047,20,0.0,0.0,0.0,0.0,0.0,Benign
3,443,6,132,2,0,0.0,0.0,0.0,0.000000,0.0,...,0,256,-1,20,0.0,0.0,0.0,0.0,0.0,Benign
4,443,6,274016,9,13,1285.0,517.0,0.0,142.777778,1460.0,...,0,8192,1047,20,0.0,0.0,0.0,0.0,0.0,Benign


In [6]:
if LABEL_COL not in df.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL}")

print("Columna objetivo encontrada correctamente.")
print()
print("Distribución global de clases:")
display(resumen_clases(df, LABEL_COL))

Columna objetivo encontrada correctamente.

Distribución global de clases:


,count,percentage
LABEL,,
Benign,10630624,88.7185
Bot,144535,1.2062
Brute Force -Web,555,0.0046
Brute Force -XSS,228,0.0019
DDOS attack-HOIC,198861,1.6596
DDOS attack-LOIC-UDP,1730,0.0144
DDoS attacks-LOIC-HTTP,575364,4.8017
DoS attacks-GoldenEye,41406,0.3456
DoS attacks-Hulk,145199,1.2118


In [7]:
train_df, test_df = dividir_train_test_stratified(
    df=df,
    label_col=LABEL_COL,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("Forma train:", train_df.shape)
print("Forma test:", test_df.shape)

Forma train: (9585940, 51)
Forma test: (2396485, 51)


In [8]:
print("Distribución de clases en TRAIN:")
display(resumen_clases(train_df, LABEL_COL))

Distribución de clases en TRAIN:


,count,percentage
LABEL,,
Benign,8504499,88.7185
Bot,115628,1.2062
Brute Force -Web,444,0.0046
Brute Force -XSS,182,0.0019
DDOS attack-HOIC,159089,1.6596
DDOS attack-LOIC-UDP,1384,0.0144
DDoS attacks-LOIC-HTTP,460291,4.8017
DoS attacks-GoldenEye,33125,0.3456
DoS attacks-Hulk,116159,1.2118


In [9]:
print("Distribución de clases en TEST:")
display(resumen_clases(test_df, LABEL_COL))

Distribución de clases en TEST:


,count,percentage
LABEL,,
Benign,2126125,88.7185
Bot,28907,1.2062
Brute Force -Web,111,0.0046
Brute Force -XSS,46,0.0019
DDOS attack-HOIC,39772,1.6596
DDOS attack-LOIC-UDP,346,0.0144
DDoS attacks-LOIC-HTTP,115073,4.8017
DoS attacks-GoldenEye,8281,0.3455
DoS attacks-Hulk,29040,1.2118


In [10]:
resumen_global = resumen_clases(df, LABEL_COL).rename(
    columns={"count": "global_count", "percentage": "global_percentage"}
)

resumen_train = resumen_clases(train_df, LABEL_COL).rename(
    columns={"count": "train_count", "percentage": "train_percentage"}
)

resumen_test = resumen_clases(test_df, LABEL_COL).rename(
    columns={"count": "test_count", "percentage": "test_percentage"}
)

comparacion = pd.concat([resumen_global, resumen_train, resumen_test], axis=1)

print("Comparación global / train / test:")
display(comparacion)

Comparación global / train / test:


,global_count,global_percentage,train_count,train_percentage,test_count,test_percentage
LABEL,,,,,,
Benign,10630624,88.7185,8504499,88.7185,2126125,88.7185
Bot,144535,1.2062,115628,1.2062,28907,1.2062
Brute Force -Web,555,0.0046,444,0.0046,111,0.0046
Brute Force -XSS,228,0.0019,182,0.0019,46,0.0019
DDOS attack-HOIC,198861,1.6596,159089,1.6596,39772,1.6596
DDOS attack-LOIC-UDP,1730,0.0144,1384,0.0144,346,0.0144
DDoS attacks-LOIC-HTTP,575364,4.8017,460291,4.8017,115073,4.8017
DoS attacks-GoldenEye,41406,0.3456,33125,0.3456,8281,0.3455
DoS attacks-Hulk,145199,1.2118,116159,1.2118,29040,1.2118


In [11]:
guardar_dataset_csv(
    df=train_df,
    nombre_archivo=NOMBRE_TRAIN,
    ruta=RUTA_SALIDA
)

guardar_dataset_csv(
    df=test_df,
    nombre_archivo=NOMBRE_TEST,
    ruta=RUTA_SALIDA
)

print("Train guardado en:")
print(RUTA_SALIDA / NOMBRE_TRAIN)
print()
print("Test guardado en:")
print(RUTA_SALIDA / NOMBRE_TEST)

Train guardado en:
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/CIC18__split__v1/CIC18__split__v1__train.csv

Test guardado en:
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/CIC18__split__v1/CIC18__split__v1__test.csv


In [12]:
reporte_split = {
    "dataset_entrada": NOMBRE_DATASET_LIMPIO,
    "label_column": LABEL_COL,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "shape_global": {
        "rows": int(df.shape[0]),
        "cols": int(df.shape[1])
    },
    "shape_train": {
        "rows": int(train_df.shape[0]),
        "cols": int(train_df.shape[1])
    },
    "shape_test": {
        "rows": int(test_df.shape[0]),
        "cols": int(test_df.shape[1])
    },
    "class_distribution_global": {
        str(k): int(v) for k, v in df[LABEL_COL].value_counts(dropna=False).to_dict().items()
    },
    "class_distribution_train": {
        str(k): int(v) for k, v in train_df[LABEL_COL].value_counts(dropna=False).to_dict().items()
    },
    "class_distribution_test": {
        str(k): int(v) for k, v in test_df[LABEL_COL].value_counts(dropna=False).to_dict().items()
    }
}

report_path = RUTA_SALIDA / NOMBRE_REPORTE

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(reporte_split, f, indent=2, ensure_ascii=False)

print("Reporte guardado en:")
print(report_path)

Reporte guardado en:
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/CIC18__split__v1/CIC18__split__v1_report.json


In [13]:
print("========== RESUMEN SPLIT ==========")
print(f"Dataset de entrada: {NOMBRE_DATASET_LIMPIO}")
print(f"Forma global: {df.shape}")
print(f"Forma train: {train_df.shape}")
print(f"Forma test: {test_df.shape}")
print(f"Test size: {TEST_SIZE}")
print(f"Random state: {RANDOM_STATE}")
print("===================================")

========== RESUMEN SPLIT ==========
Dataset de entrada: CIC18__cleanning__v1.csv
Forma global: (11982425, 51)
Forma train: (9585940, 51)
Forma test: (2396485, 51)
Test size: 0.2
Random state: 42
